In [1]:
from tensorflow.keras.utils import to_categorical
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

%matplotlib inline
import tensorflow as tf

import os

2025-11-22 12:10:29.819039: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-22 12:10:29.855470: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-22 12:10:29.855507: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-22 12:10:29.856547: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-22 12:10:29.862203: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-22 12:10:29.862755: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [15]:
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import plot_model

model = load_model('../../../documents/Benchmarks/Toptag/qkeras/lstm/model_toptag_lstm.h5')

import pprint
print(model.get_weights()[2].shape)

(80,)


In [5]:
import hls4ml


config = hls4ml.utils.config_from_keras_model(model, granularity='model', default_precision='ap_fixed<16,6>')

hls_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=config, output_dir='model_lstm_hls/hls4ml_prj', part='xc7vx690tffg1761-2'
)

In [6]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file="toptagging_lstm_layers.png")
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer1 (LSTM)               (None, 20)                2160      
                                                                 
 layer3 (Dense)              (None, 64)                1344      
                                                                 
 output_sigmoid (Dense)      (None, 1)                 65        
                                                                 
Total params: 3569 (13.94 KB)
Trainable params: 3569 (13.94 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [4]:
hls_model.compile()

In [6]:
os.environ["XILINX_VIVADO"] = "/tools/Disk_Xilinx/Vivado/2019.1"
os.environ["XILINXD_LICENSE_FILE"] = "2100@xilinxd.xilinx-dev"
os.environ["PATH"] = "/tools/Disk_Xilinx/Vivado/2019.1/bin:" + os.environ.get("PATH", "")
hls_model.build(csim=False, synth=True, vsynth=True)


****** Vivado(TM) HLS - High-Level Synthesis from C, C++ and SystemC v2019.1 (64-bit)
  **** SW Build 2552052 on Fri May 24 14:47:09 MDT 2019
  **** IP Build 2548770 on Fri May 24 18:01:18 MDT 2019
    ** Copyright 1986-2019 Xilinx, Inc. All Rights Reserved.

source /tools/Disk_Xilinx/Vivado/2019.1/scripts/vivado_hls/hls.tcl -notrace
INFO: [HLS 200-10] Running '/tools/Disk_Xilinx/Vivado/2019.1/bin/unwrapped/lnx64.o/vivado_hls'
INFO: [HLS 200-10] For user 'leo' on host 'hvm1' (Linux_x86_64 version 5.15.0-161-generic) on Sat Nov 22 12:10:56 PST 2025
INFO: [HLS 200-10] On os Ubuntu 22.04.5 LTS
INFO: [HLS 200-10] In directory '/home/leo/HLS4ML_VS_MANUAL/documents/Benchmarks/Toptag/model_lstm_hls/hls4ml_prj'
Sourcing Tcl script 'build_prj.tcl'
INFO: [HLS 200-10] Creating and opening project '/home/leo/HLS4ML_VS_MANUAL/documents/Benchmarks/Toptag/model_lstm_hls/hls4ml_prj/myproject_prj'.
INFO: [HLS 200-10] Adding design file 'firmware/myproject.cpp' to the project
INFO: [HLS 200-10] Adding 

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '4.890',
  'BestLatency': '353',
  'WorstLatency': '353',
  'IntervalMin': '354',
  'IntervalMax': '354',
  'BRAM_18K': '41',
  'DSP': '2498',
  'FF': '76158',
  'LUT': '116669',
  'URAM': '0',
  'AvailableBRAM_18K': '2940',
  'AvailableDSP': '3600',
  'AvailableFF': '866400',
  'AvailableLUT': '433200',
  'AvailableURAM': '0'},
 'VivadoSynthReport': {'LUT': '59224',
  'FF': '37062',
  'BRAM_18K': '20.5',
  'DSP48E': '2489'}}

In [7]:
hls4ml.report.read_vivado_report('model_lstm_hls/hls4ml_prj')

Found 1 solution(s) in model_lstm_hls/hls4ml_prj/myproject_prj.
Reports for solution "solution1":

C simulation report not found.
SYNTHESIS REPORT:
== Vivado HLS Report for 'myproject'
* Date:           Sat Nov 22 12:20:48 2025

* Version:        2019.1 (Build 2552052 on Fri May 24 15:28:33 MDT 2019)
* Project:        myproject_prj
* Solution:       solution1
* Product family: virtex7
* Target device:  xc7vx690t-ffg1761-2


== Performance Estimates
+ Timing (ns): 
    * Summary: 
    +--------+-------+----------+------------+
    |  Clock | Target| Estimated| Uncertainty|
    +--------+-------+----------+------------+
    |ap_clk  |   5.00|     4.890|        0.62|
    +--------+-------+----------+------------+

+ Latency (clock cycles): 
    * Summary: 
    +-----+-----+-----+-----+---------+
    |  Latency  |  Interval | Pipeline|
    | min | max | min | max |   Type  |
    +-----+-----+-----+-----+---------+
    |  353|  353|  353|  353|   none  |
    +-----+-----+-----+-----+-------